In [1]:
# Install Required Libraries
!pip install -q sentence-transformers faiss-cpu transformers sentencepiece

In [2]:
# Create Synthetic Company Documents

documents = {
    "leave_policy.txt": """
    Company Leave Policy

    Employees receive 20 paid leaves annually.
    Employees can carry forward a maximum of 5 unused
    leaves to the following year.

    Sick leave must be reported to the manager.
    Employees must submit leave requests at least
    3 working days in advance for planned leave.
    """,

    "work_from_home_policy.txt": """
    Work From Home Policy

    Employees can work from home up to 2 days per week.
    Employees must obtain manager approval before
    working remotely.

    Employees must remain available during core working
    hours from 10 AM to 4 PM.

    All employees must attend mandatory office meetings
    when requested.
    """,

    "reimbursement_policy.txt": """
    Employee Reimbursement Policy

    Employees can claim work-related travel expenses
    up to 15000 rupees per month.

    Meal expenses are reimbursable up to 500 rupees
    per working day.

    Employees must submit original receipts along
    with reimbursement claims.

    Reimbursement requests must be submitted within
    30 days of the expense.
    """
}

# Save documents as text files
import os

os.makedirs("company_docs", exist_ok=True)

for filename, content in documents.items():
    with open(f"company_docs/{filename}", "w") as file:
        file.write(content.strip())

print("Documents created successfully!")

print("\nAvailable documents:")
print(os.listdir("company_docs"))

Documents created successfully!

Available documents:
['work_from_home_policy.txt', 'leave_policy.txt', 'reimbursement_policy.txt']


In [3]:
# Load documents and split them into smaller chunks

documents_list = []

for filename in os.listdir("company_docs"):
    file_path = os.path.join("company_docs", filename)

    with open(file_path, "r") as file:
        content = file.read()

    documents_list.append({
        "source": filename,
        "content": content
    })

# Split documents into chunks
chunk_size = 200
overlap = 40

chunks = []

for document in documents_list:
    content = document["content"]

    start = 0

    while start < len(content):
        end = start + chunk_size

        chunk_text = content[start:end]

        chunks.append({
            "text": chunk_text,
            "source": document["source"]
        })

        start += chunk_size - overlap

print("Total documents:", len(documents_list))
print("Total chunks:", len(chunks))

# Display generated chunks
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i + 1} ({chunk['source']}):")
    print(chunk["text"])

Total documents: 3
Total chunks: 7

Chunk 1 (work_from_home_policy.txt):
Work From Home Policy

    Employees can work from home up to 2 days per week.
    Employees must obtain manager approval before
    working remotely.

    Employees must remain available during core 

Chunk 2 (work_from_home_policy.txt):
oyees must remain available during core working
    hours from 10 AM to 4 PM.

    All employees must attend mandatory office meetings
    when requested.

Chunk 3 (leave_policy.txt):
Company Leave Policy

    Employees receive 20 paid leaves annually.
    Employees can carry forward a maximum of 5 unused
    leaves to the following year.

    Sick leave must be reported to the man

Chunk 4 (leave_policy.txt):
  Sick leave must be reported to the manager.
    Employees must submit leave requests at least
    3 working days in advance for planned leave.

Chunk 5 (reimbursement_policy.txt):
Employee Reimbursement Policy

    Employees can claim work-related travel expenses
    up t

In [4]:
# Generate embeddings and create a FAISS vector index

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract chunk text
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate embeddings
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True
)

# Create FAISS index
dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(embeddings.astype("float32"))

print("Embedding shape:", embeddings.shape)
print("FAISS index created successfully!")
print("Total vectors stored:", faiss_index.ntotal)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (7, 384)
FAISS index created successfully!
Total vectors stored: 7


In [6]:
# Retrieve the top 3 relevant document chunks for a question

def retrieve_documents(question, top_k=2):

    # Convert question into an embedding
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    # Search FAISS index
    distances, indices = faiss_index.search(
        question_embedding.astype("float32"),
        top_k
    )

    # Collect retrieved chunks
    retrieved_chunks = []

    for index in indices[0]:
        if index != -1:
            retrieved_chunks.append(chunks[index])

    return retrieved_chunks


# Test document retrieval
question = "How many paid leaves do employees receive annually?"

results = retrieve_documents(question)

print("Question:", question)
print("\nRetrieved documents:")

for i, result in enumerate(results):
    print(f"\nResult {i + 1}:")
    print("Source:", result["source"])
    print("Content:", result["text"])


Question: How many paid leaves do employees receive annually?

Retrieved documents:

Result 1:
Source: leave_policy.txt
Content: Company Leave Policy

    Employees receive 20 paid leaves annually.
    Employees can carry forward a maximum of 5 unused
    leaves to the following year.

    Sick leave must be reported to the man

Result 2:
Source: reimbursement_policy.txt
Content: Employee Reimbursement Policy

    Employees can claim work-related travel expenses
    up to 15000 rupees per month.

    Meal expenses are reimbursable up to 500 rupees
    per working day.

    Emp


In [10]:
# Load FLAN-T5 and generate answers using retrieved documents

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model and tokenizer
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("FLAN-T5 loaded successfully!")
print("Device:", device)


# Generate an answer using retrieved documents
def generate_answer(question, retrieved_chunks):

    context = "\n".join(
        [chunk["text"] for chunk in retrieved_chunks]
    )

    prompt = f"""
    Answer the question using only the provided context.
    If the answer is not available, say:
    Information not found in the documents.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    # Generate answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


# Test answer generation
question = "How many paid leaves do employees receive annually?"

results = retrieve_documents(question, top_k=2)

answer = generate_answer(question, results)

print("Question:", question)
print("\nGenerated Answer:", answer)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully!
Device: cpu
Question: How many paid leaves do employees receive annually?

Generated Answer: 20


In [11]:

# Build a RAG assistant that answers questions with source citations

def rag_assistant(question):

    # Retrieve relevant document chunks
    retrieved_chunks = retrieve_documents(question, top_k=2)

    # Generate answer using retrieved context
    answer = generate_answer(question, retrieved_chunks)

    # Collect source document names
    sources = list(set(
        chunk["source"] for chunk in retrieved_chunks
    ))

    print("Question:", question)
    print("\nAnswer:", answer)
    print("\nSources:")

    for source in sources:
        print("-", source)


# Test the RAG assistant
rag_assistant("How many paid leaves do employees receive annually?")

Question: How many paid leaves do employees receive annually?

Answer: 20

Sources:
- reimbursement_policy.txt
- leave_policy.txt


In [12]:
# Evaluate the RAG assistant using five predefined questions

evaluation_questions = [
    {
        "question": "How many paid leaves do employees receive annually?",
        "expected": "20"
    },
    {
        "question": "How many unused leaves can employees carry forward?",
        "expected": "5"
    },
    {
        "question": "How many days per week can employees work from home?",
        "expected": "2"
    },
    {
        "question": "What is the monthly limit for travel expense reimbursement?",
        "expected": "15000"
    },
    {
        "question": "How many days do employees have to submit reimbursement claims?",
        "expected": "30"
    }
]

# Run evaluation
evaluation_results = []

for item in evaluation_questions:

    retrieved_chunks = retrieve_documents(
        item["question"], top_k=2
    )

    answer = generate_answer(
        item["question"], retrieved_chunks
    )

    evaluation_results.append({
        "question": item["question"],
        "expected": item["expected"],
        "generated": answer,
        "sources": ", ".join(
            set(chunk["source"] for chunk in retrieved_chunks)
        )
    })

# Display evaluation results
import pandas as pd

results_df = pd.DataFrame(evaluation_results)

print("RAG EVALUATION RESULTS")
display(results_df)


RAG EVALUATION RESULTS


,question,expected,generated,sources
0,How many paid leaves do employees receive annu...,20,20,"reimbursement_policy.txt, leave_policy.txt"
1,How many unused leaves can employees carry for...,5,5,"reimbursement_policy.txt, leave_policy.txt"
2,How many days per week can employees work from...,2,2,work_from_home_policy.txt
3,What is the monthly limit for travel expense r...,15000,15000 rupees,reimbursement_policy.txt
4,How many days do employees have to submit reim...,30,30,reimbursement_policy.txt
